The goal is to generate a dataset of video's frame according to parameters.
The dataset is the same dataset use for png treatment. So it's based on png2.txt which refer all png's files use to train and validate.

In [ ]:
import numpy as np
import os
import cv2

from pathlib import Path

In [ ]:
def get_videos_file(png_file: str, output_file: str):
    """
    Read png_file that contain all the png and try to get the video associeted.
    Write in an output file
    :param png_file: The png file
    :param output_file: Write the path of all the video that can be open and read
    """
    vid_folders = 'avi'
    
    output = open(output_file, "a")
    
    with open(png_file) as file:
        lines = file.readlines()
        for line in lines:
            
            splited = line.split('\\')
            png_name = splited[-1] 
            vid_name = png_name.split('.')[0] + '.avi'
            
            # remove the path to png
            folders = splited[:-2]
            
            # append the path to the video
            folders.append(vid_folders)
            folders.append(vid_name)
            path_video = '\\'.join(folders)
            
            # write the path of the video
            if Path(path_video).exists():
                output.write(path_video + '\n')
            else:
                print(f"Failed to write, {path_video} does not exist !")
        
    output.close()

In [ ]:
def _split_vid(video_path: str, nb_frame):
    
    video = cv2.VideoCapture(video_path)
    if not video.isOpened():
        print(f"Failed to open {video_path} as a video")
        return []
    
    # information from the video
    frame_per_chunk = int(video.get(cv2.CAP_PROP_FRAME_COUNT) // nb_frame)
    height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    result = []
    
    ret = True
        
    # read a full video
    while ret:
        
        average_frame = np.zeros((height, width), dtype=np.double)
        nb = 0
        for _ in range(frame_per_chunk):
            ret, frame = video.read()
            if not ret:
                break
            
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            nb += 1
            average_frame += gray
        
        if ret:    
            average_frame /= nb
            result.append(average_frame)
        
    video.release()
    return result
        

def extract_frames(video_file, output_folder, nb_frame_result = 8):
    
    # create output folder
    os.makedirs(output_folder, exist_ok=True)
    
    with open(video_file) as file:
        for line in file:
            
            # remove space and other special char
            video_path = line.strip()
            
            _, video_name = os.path.split(video_path)
            video_name_without_ext, _ = os.path.splitext(video_name)
            
            lst_result_frames = _split_vid(video_path, nb_frame_result)
            
            if len(lst_result_frames) == 0:
                print(f"Failed to extract frames from {video_name}")
                continue
            
            # create the folder of the video
            folder_vid = os.path.join(output_folder, video_name_without_ext)
            os.makedirs(folder_vid, exist_ok=True)

            # save all frame in the folder of the video
            for i, frame in enumerate(lst_result_frames):
                filename = os.path.join(folder_vid, f"{i:04d}.png")
                cv2.imwrite(filename, frame)
            
            
    return True

In [ ]:
png_dataset_file = "pngs2.txt"
avi_dataset_file = "avi.txt"

get_videos_file(png_dataset_file, avi_dataset_file)
extract_frames(avi_dataset_file, "dataset_video")